# Retail Sales & Profit Intelligence — Data Cleaning & Fact Table Build
Dataset: Olist Brazilian E-Commerce Public Dataset (9 CSVs)

This notebook cleans the raw Olist tables, builds a fact table at order-item grain, simulates a cost/profit layer (the raw data has no cost column), and produces a star-schema set of tables ready for Power BI.

## Step 1: Load raw data

In [1]:
import pandas as pd
import numpy as np

data_path = "../data/raw/"

customers = pd.read_csv(data_path + "olist_customers_dataset.csv")
geolocation = pd.read_csv(data_path + "olist_geolocation_dataset.csv")
order_items = pd.read_csv(data_path + "olist_order_items_dataset.csv")
payments = pd.read_csv(data_path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(data_path + "olist_order_reviews_dataset.csv")
orders = pd.read_csv(data_path + "olist_orders_dataset.csv")
products = pd.read_csv(data_path + "olist_products_dataset.csv")
sellers = pd.read_csv(data_path + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(data_path + "product_category_name_translation.csv")

datasets = {
    "customers": customers, "geolocation": geolocation, "order_items": order_items,
    "payments": payments, "reviews": reviews, "orders": orders,
    "products": products, "sellers": sellers, "category_translation": category_translation
}
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (1000163, 5)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


## Step 2: Data quality audit
Check order status distribution, nulls, duplicates, and category coverage before cleaning.

In [2]:
print("ORDER STATUS BREAKDOWN")
print(orders['order_status'].value_counts())

print("\nDUPLICATE order_id in orders:", orders['order_id'].duplicated().sum())
print("DUPLICATE order_id+order_item_id in items:", order_items.duplicated(subset=['order_id','order_item_id']).sum())

missing_cats = set(products['product_category_name'].dropna()) - set(category_translation['product_category_name'])
print("\nCategories missing from translation table:", missing_cats)

print("\nPayment rows with value <= 0:", (payments['payment_value'] <= 0).sum())

orders_no_items = set(orders['order_id']) - set(order_items['order_id'])
print("\nOrders with no line items:", len(orders_no_items))
print(orders[orders['order_id'].isin(orders_no_items)]['order_status'].value_counts())

ORDER STATUS BREAKDOWN
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

DUPLICATE order_id in orders: 0
DUPLICATE order_id+order_item_id in items: 0

Categories missing from translation table: {'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}

Payment rows with value <= 0: 9

Orders with no line items: 775


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


## Step 3: Clean dates, categories, and payments
- Convert date columns to real datetimes (nulls kept as NaT — they reflect real order states, e.g. a cancelled order never gets a delivery date)
- Fill the 2 categories missing from the translation table
- Fill missing product category as `unknown`
- Add delivery performance features (delivery days, delay vs. estimate, is_late)
- Drop the handful of zero-value payment rows

In [3]:
date_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
             'order_delivered_customer_date','order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

missing_map = {
    'portateis_cozinha_e_preparadores_de_alimentos': 'kitchen_portables_and_food_preparers',
    'pc_gamer': 'pc_gamer'
}
extra_rows = pd.DataFrame(list(missing_map.items()), columns=['product_category_name','product_category_name_english'])
category_translation_full = pd.concat([category_translation, extra_rows], ignore_index=True)

products['product_category_name'] = products['product_category_name'].fillna('unknown')
products = products.merge(category_translation_full, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')

orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['delivery_delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days
orders['is_late'] = orders['delivery_delay_days'] > 0

print("Late delivery rate (delivered orders):", orders.loc[orders['order_status']=='delivered','is_late'].mean().round(3))

payments_clean = payments[payments['payment_value'] > 0].copy()
print("Payment rows dropped:", len(payments) - len(payments_clean))

Late delivery rate (delivered orders): 0.068
Payment rows dropped: 9


## Step 4: Build the fact table (order-item grain)
Join items → orders → products → customers → sellers → aggregated payments into one fact table.

In [4]:
payments_agg = payments_clean.groupby('order_id').agg(
    payment_value=('payment_value','sum'),
    payment_type=('payment_type', lambda x: x.mode()[0] if not x.mode().empty else 'unknown'),
    payment_installments=('payment_installments','max')
).reset_index()

fact = order_items.merge(orders, on='order_id', how='inner')
fact = fact.merge(products[['product_id','product_category_name_english','product_weight_g']], on='product_id', how='left')
fact = fact.merge(customers[['customer_id','customer_city','customer_state']], on='customer_id', how='left')
fact = fact.merge(sellers[['seller_id','seller_city','seller_state']], on='seller_id', how='left')
fact = fact.merge(payments_agg, on='order_id', how='left')
fact['product_category_name_english'] = fact['product_category_name_english'].fillna('unknown')

print("Fact table shape:", fact.shape)
fact.isnull().sum()[fact.isnull().sum() > 0]

Fact table shape: (112650, 26)


order_approved_at                  15
order_delivered_carrier_date     1194
order_delivered_customer_date    2454
delivery_days                    2454
delivery_delay_days              2454
product_weight_g                   18
payment_value                       3
payment_type                        3
payment_installments                3
dtype: int64

## Step 5: Simulate cost & profit
The raw Olist data has `price` and `freight_value` only — no cost basis, so no real profit can be computed. We simulate a realistic cost layer: each category gets a base cost ratio (60–75% of price), then each individual transaction gets random noise (±5 points, clipped to 45–88%) so margins vary transaction-to-transaction instead of being artificially flat.

**This is simulated data for portfolio/demo purposes — noted clearly in the README.**

In [5]:
np.random.seed(42)  # reproducible

categories = fact['product_category_name_english'].unique()
category_base_ratio = {cat: np.random.uniform(0.60, 0.75) for cat in categories}
fact['base_cost_ratio'] = fact['product_category_name_english'].map(category_base_ratio)

noise = np.random.normal(loc=0, scale=0.05, size=len(fact))
fact['cost_ratio'] = (fact['base_cost_ratio'] + noise).clip(0.45, 0.88)

fact['cost'] = (fact['price'] * fact['cost_ratio']).round(2)
fact['profit'] = (fact['price'] - fact['cost']).round(2)
fact['profit_margin_pct'] = ((fact['profit'] / fact['price']) * 100).round(2)
fact = fact.drop(columns=['base_cost_ratio'])

print("Overall margin %:", (fact['profit'].sum() / fact['price'].sum() * 100).round(2))
fact[['price','cost','profit','profit_margin_pct']].describe()

Overall margin %: 33.27


,price,cost,profit,profit_margin_pct
count,112650.000000,112650.000000,112650.000000,112650.000000
mean,120.653739,80.507126,40.146613,33.273047
std,183.633928,122.811261,63.452079,6.897188
min,0.850000,0.490000,0.210000,11.990000
25%,39.900000,26.650000,12.770000,28.450000
50%,74.990000,49.620000,23.980000,33.430000
75%,134.900000,89.480000,44.920000,38.190000
max,6735.000000,4210.050000,2875.280000,55.010000


## Step 6: Build star schema (dimensions + rollups) for Power BI

In [6]:
min_date = fact['order_purchase_timestamp'].min().normalize()
max_date = fact['order_purchase_timestamp'].max().normalize()
dim_date = pd.DataFrame({'date': pd.date_range(min_date, max_date, freq='D')})
dim_date['year'] = dim_date['date'].dt.year
dim_date['month'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.strftime('%b')
dim_date['year_month'] = dim_date['date'].dt.strftime('%Y-%m')
dim_date['quarter'] = dim_date['date'].dt.quarter
dim_date['day_of_week'] = dim_date['date'].dt.day_name()
dim_date['is_weekend'] = dim_date['date'].dt.dayofweek >= 5

dim_products = products[['product_id','product_category_name_english','product_weight_g',
                          'product_length_cm','product_height_cm','product_width_cm']].drop_duplicates(subset='product_id')
dim_products = dim_products.rename(columns={'product_category_name_english':'category'})

dim_customers = customers[['customer_id','customer_unique_id','customer_city','customer_state']].drop_duplicates(subset='customer_id')
dim_sellers = sellers[['seller_id','seller_city','seller_state']].drop_duplicates(subset='seller_id')

fact['year_month'] = fact['order_purchase_timestamp'].dt.strftime('%Y-%m')
monthly_rollup = fact.groupby('year_month').agg(
    orders=('order_id','nunique'), items_sold=('order_item_id','count'),
    revenue=('price','sum'), freight=('freight_value','sum'), profit=('profit','sum')
).reset_index()
monthly_rollup['margin_pct'] = (monthly_rollup['profit']/monthly_rollup['revenue']*100).round(2)

category_rollup = fact.groupby('product_category_name_english').agg(
    items_sold=('order_item_id','count'), revenue=('price','sum'), profit=('profit','sum')
).reset_index().sort_values('revenue', ascending=False)
category_rollup['margin_pct'] = (category_rollup['profit']/category_rollup['revenue']*100).round(2)

print("Note: 2018-09 has only 1 order — dataset ends mid-month, exclude/annotate in dashboard.")
category_rollup.head(10)

Note: 2018-09 has only 1 order — dataset ends mid-month, exclude/annotate in dashboard.


,product_category_name_english,items_sold,revenue,profit,margin_pct
43,health_beauty,9670,1258681.34,340763.52,27.07
73,watches_gifts,5991,1205005.68,446896.40,37.09
7,bed_bath_table,11115,1036988.68,411343.62,39.67
67,sports_leisure,8641,988048.97,250735.60,25.38
15,computers_accessories,7827,911954.32,306346.18,33.59
39,furniture_decor,8334,729762.49,212479.02,29.12
20,cool_stuff,3796,635290.85,219063.01,34.48
49,housewares,6964,632248.66,238404.95,37.71
5,auto,4235,592720.11,192388.43,32.46
42,garden_tools,4347,485256.46,181719.09,37.45


## Step 7: Export cleaned tables for Power BI / SQL validation

In [7]:
fact.to_csv("../data/processed/fact_sales.csv", index=False)
dim_date.to_csv("../data/processed/dim_date.csv", index=False)
dim_products.to_csv("../data/processed/dim_products.csv", index=False)
dim_customers.to_csv("../data/processed/dim_customers.csv", index=False)
dim_sellers.to_csv("../data/processed/dim_sellers.csv", index=False)
monthly_rollup.to_csv("../data/processed/monthly_rollup.csv", index=False)
category_rollup.to_csv("../data/processed/category_rollup.csv", index=False)

print("All processed tables saved to data/processed/")

All processed tables saved to data/processed/
